# Previsão do sentimento de consumidores de e-commerce brasileiro a partir de suas avaliações — Regressão Logística

## Contexto

Com a constante evolução tecnológica, o e-commerce é utilizado por muitos brasileiros pela facilidade de compra e pelos preços possivelmente reduzidos. Com isso, surge a curiosidade em compreender como esses consumidores realmente se sentem após a compra: satisfeitos com o produto (de acordo com os comentários positivos) ou insatisfeitos (de acordo com os comentários negativos). Dependendo dos dados obtidos e do nível de satisfação do cliente, é possível identificar a proporção de cada sentimento e quais palavras são mais recorrentes, o que pode indicar de onde vem cada sentimento e orientar melhorias na gestão de produtos.

## Objetivo

O objetivo deste projeto é dar início aos estudos de Aprendizado de Máquina, utilizando dados de um dataset da plataforma Kaggle chamado Merged Olist Brazilian E-Commerce Dataset for EDA (Conjunto de Dados de E-Commerce Brasileiro da Olist Mesclado para EDA), criado para fins educacionais.

Com ele, vou treinar um modelo supervisionado de regressão logística que, com base apenas no comentário do cliente (cada linha representa um pedido), preveja se ele ficou Satisfeito, Neutro ou Insatisfeito após receber a compra. Dessa forma, o modelo poderá generalizar o que aprendeu e classificar novos dados, caso eles cheguem sem rótulo.

Além do treinamento, vou testar o modelo e calcular as seguintes métricas de avaliação:

- Acurácia: mede a proporção de previsões corretas.
- Precisão: avalia a proporção de verdadeiros positivos em relação ao total de positivos previstos.
- Recall: indica a capacidade do modelo de identificar todos os casos positivos.
- F1-score: combina precisão e recall em uma única métrica.

Por fim, vou levantar as palavras mais frequentes nos comentários, para tentar identificar o que levou às avaliações negativas, e realizar algumas análises de estatística descritiva para resumir os resultados obtidos. 

## Entrada e saída do modelo

- **Entrada:** o comentário do cliente (`review_comment_message`), convertido em valores numéricos por meio de Bag of Words.
- **Saída:** o rótulo de satisfação (Satisfeito, Neutro ou Insatisfeito).

## Criação dos rótulos

Além do comentário, cada cliente fornece uma nota de avaliação (`review_score`) de 1 a 5. A partir dela, criei os rótulos de satisfação com a seguinte regra:

| Score | Rótulo |
|---|---|
| 5 e 4 | Satisfeito |
| 3 | Neutro |
| 2 e 1 | Insatisfeito |

O score **não é usado como entrada**, pois isso entregaria a resposta ao modelo. Ele é usado apenas para gerar os rótulos que servem de gabarito durante o treinamento.

## Métodos de validação

Para consolidação dos conhecimentos, dois métodos de validação foram aplicados:

- Validação **holdout**: Os dados foram divididos em treino (80%) e teste (20%). O modelo aprende com os comentários e rótulos do conjunto de treino e, em seguida, prevê os rótulos dos comentários do conjunto de teste, que ele nunca viu. Esses rótulos previstos são comparados com os rótulos reais, obtidos a partir dos scores fornecidos pelos clientes, para verificar se condizem ou não com o resultado esperado.

- Validação **cruzada**: Os dados foram divididos em várias partes, e o modelo foi treinado e testado em diferentes combinações dessas partes. 

## Pré-processamento dos dados: Limpeza

In [1]:
# Importa bibliotecas necessárias
import pandas as pd
pd.set_option('display.max_colwidth', None)
import re # operações regex
import nltk # processamento de linguagem natural (PLN)

# biblioteca de machine learning 
from sklearn.feature_extraction.text import CountVectorizer # Vetorização da Linguagem Natural
from sklearn.linear_model import LogisticRegression # Aplicação da Regressão Logística
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict # Validações
from sklearn.metrics import classification_report, confusion_matrix # Resultado das validações
from sklearn.pipeline import Pipeline # Para fazer o processo de vetorização e análise após a separação dos dados

nltk.download('stopwords') # lista de stopwords de vários idiomas
nltk.download('punkt_tab') # modelo treinado que sabe onde terminam frases e palavras

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/baee7ae1-eb79-4aaa-9a44-
[nltk_data]     1f43d328681d/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/baee7ae1-eb79-4aaa-9a44-
[nltk_data]     1f43d328681d/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
# Cria DataFrame
def cria_df(csv, sep):
    
    df = pd.read_csv(csv, sep=sep)
    
    print('Primeira visualização dos dados:')
    display(df.head(30))

    print(40 * '-')

    linhas, colunas = df.shape
    print(f"Os dados possuem {linhas} linhas e {colunas} colunas.")

    print(40 * '-')
    
    tipo_dados = df.dtypes
    print(f"Tipo de dados para cada coluna: \n{tipo_dados}")

    print(40 * '-')
    
    valores_nulos = round((df.isnull().sum() / linhas) * 100, 2)
    print("Proporção de valores nulos por coluna:")
    print(valores_nulos)

    print(40 * '-')
    
    duplicatas = df.duplicated().sum()
    print(f"Quantidade de linhas duplicadas: {duplicatas}")

    print(40 * '-')
    
    return df, valores_nulos
    

In [3]:
# Realiza limpeza inicial dos dados
def limpa_dados(df, sep):

    # Verifica as características dos dados e guarda os valores das proporções de nulos.
    df, valores_nulos = cria_df(df, sep=sep)

    # Remove duplicatas
    print('Removendo duplicatas...')
    df_sem_duplicatas = df.drop_duplicates(keep='first') # Mantém a primeira linha do registro

    print(40 * '-')

    
    print(f'Verificando se ainda há duplicatas: {df_sem_duplicatas.duplicated().sum()}')

    print(40 * '-')

    # Trata valores nulos
    # Critério que adotei: colunas com a partir de 70% dos valores nulos, removo
    print('Tratando colunas com valores nulos...')
    colunas_nulas = valores_nulos[valores_nulos >= 70].index # extrai o nome das colunas
    df_sem_nulos = df_sem_duplicatas.drop(columns = colunas_nulas)

    print(40 * '-')
    
    print(f'Verificando se ainda há colunas com valores nulos igual ou acima de 70%: \n{df_sem_nulos.isnull().sum()/len(df_sem_nulos)}')

    print(40 * '-')

    # Mostra como os dados estão no momento
    print('Como os dados estão no momento:')
    display(df_sem_nulos.head(30))

    print(40 * '-')

    return df_sem_nulos

In [4]:
# Seleciona variáveis que vão ser usadas no treinamento
dados_limpos = limpa_dados('olist_merged_dataset.csv', ',')

print('Selecionando variáveis pro treinamento...')

print(40 * '-')

dados_limpos = dados_limpos.drop(columns=[
                                          'customer_id',
                                          'order_status', 
                                          'order_purchase_timestamp',
                                          'order_approved_at',
                                          'order_delivered_carrier_date',
                                          'order_delivered_customer_date',
                                          'order_estimated_delivery_date',
                                          'customer_unique_id',
                                          'customer_zip_code_prefix',
                                          'customer_city',
                                          'customer_state',
                                          'order_item_id',
                                          'seller_id',
                                          'shipping_limit_date',
                                          'price',
                                          'freight_value',
                                          'product_name_lenght',
                                          'product_description_lenght',
                                          'product_photos_qty',
                                          'product_weight_g',
                                          'product_length_cm',
                                          'product_height_cm',
                                          'product_width_cm',
                                          'seller_zip_code_prefix',
                                          'seller_city',
                                          'seller_state',
                                          'payment_sequential',
                                          'payment_type',
                                          'payment_installments',
                                          'payment_value',
                                          'review_id',
                                          'review_creation_date',
                                          'review_answer_timestamp',
                                          'delivery_days',
                                          'delivery_delay',
                                          'product_category_name_english'])

# Após excluir colunas que não vão ser utilizadas, confiro se há linhas duplicadas entre essas colunas
print(f'Quantidade de linhas duplicadas após a seleção: {dados_limpos.duplicated().sum()}')

print(40 * '-')

print('Removendo duplicatas...')

print(40 * '-')

dados_limpos['review_comment_message'] = dados_limpos['review_comment_message'].str.strip()
dados_limpos = dados_limpos.drop_duplicates(keep='first', subset=['order_id'])
dados_limpos = dados_limpos.dropna(subset=['review_comment_message'])
dados_limpos = dados_limpos[dados_limpos['review_comment_message'] != ''] # Removendo casos de comentários vazios 


print(f'Shape final: {dados_limpos.shape}')



# Para deixar claro:
# No primeiro momento, analisei se havia linhas duplicadas considerando todas as variáveis. Não havia.
# No segundo momento, analisei se havia linhas duplicadas considerando apenas as colunas selecionadas. Havia.
# No terceiro momento, removi as linhas duplicadas, pois um mesmo pedido pode conter vários produtos
# e, com isso, o comentário de cada pedido se repete em todos os produtos dele.

Primeira visualização dos dados:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,delivery_days,delivery_delay
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1,18.12,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1,2.00,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1,18.59,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1,141.46,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,13.0,-6.0
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,3,179.12,e73b67b67587f7644d5bd1a52deb1b01,5,NaN,NaN,2018-08-18,2018-08-22 19:07:58,9.0,-18.0
5,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1,72.20,359d03e676b3c069f62cadba8dd3f6e8,5,NaN,O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista.,2017-12-03,2017-12-05 19:21:58,13.0,-13.0
6,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1,28.62,e50934924e227544ba8246aeb3770dd4,5,NaN,NaN,2018-02-17,2018-02-18 13:02:51,2.0,-10.0
7,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,80bb27c7c16e8f973207a5086ab329e2,86320,...,6,175.26,89b738e70a1ce346db29a20fb2910161,4,NaN,NaN,2017-07-27,2017-07-27 22:48:30,16.0,-6.0
8,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,932afa1e708222e5821dac9cd5db4cae,26525,...,3,75.16,07d67dd06ed5f88bef11ef6b464e79ae,5,NaN,NaN,2017-05-27,2017-05-28 02:59:57,9.0,-12.0
9,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,39382392765b6dc74812866ee5ee92a7,99655,...,1,35.95,fc4af8aea8ec3f1a3c

----------------------------------------
Os dados possuem 115609 linhas e 42 colunas.
----------------------------------------
Tipo de dados para cada coluna: 
order_id                          object
customer_id                       object
order_status                      object
order_purchase_timestamp          object
order_approved_at                 object
order_delivered_carrier_date      object
order_delivered_customer_date     object
order_estimated_delivery_date     object
customer_unique_id                object
customer_zip_code_prefix           int64
customer_city                     object
customer_state                    object
order_item_id                      int64
product_id                        object
seller_id                         object
shipping_limit_date               object
price                            float64
freight_value                    float64
product_category_name             object
product_name_lenght              float64
product_description_

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_message,review_creation_date,review_answer_timestamp,delivery_days,delivery_delay
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,credit_card,1,18.12,a54f0611adc9ed256b57ede6b6eb5114,4,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,voucher,1,2.00,a54f0611adc9ed256b57ede6b6eb5114,4,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,voucher,1,18.59,a54f0611adc9ed256b57ede6b6eb5114,4,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,boleto,1,141.46,8d5266042046a06655c8db133d120ba5,4,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,13.0,-6.0
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,credit_card,3,179.12,e73b67b67587f7644d5bd1a52deb1b01,5,NaN,2018-08-18,2018-08-22 19:07:58,9.0,-18.0
5,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,credit_card,1,72.20,359d03e676b3c069f62cadba8dd3f6e8,5,O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista.,2017-12-03,2017-12-05 19:21:58,13.0,-13.0
6,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,credit_card,1,28.62,e50934924e227544ba8246aeb3770dd4,5,NaN,2018-02-17,2018-02-18 13:02:51,2.0,-10.0
7,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,80bb27c7c16e8f973207a5086ab329e2,86320,...,credit_card,6,175.26,89b738e70a1ce346db29a20fb2910161,4,NaN,2017-07-27,2017-07-27 22:48:30,16.0,-6.0
8,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,932afa1e708222e5821dac9cd5db4cae,26525,...,credit_card,3,75.16,07d67dd06ed5f88bef11ef6b464e79ae,5,NaN,2017-05-27,2017-05-28 02:59:57,9.0,-12.0
9,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,39382392765b6dc74812866ee5ee92a

----------------------------------------
Selecionando variáveis pro treinamento...
----------------------------------------
Quantidade de linhas duplicadas após a seleção: 15112
----------------------------------------
Removendo duplicatas...
----------------------------------------
Shape final: (39638, 5)


## Processamento de Linguagem Natural + Modelo de Regressão Logística

In [5]:
# 1. Pré-processamento Linguagem Natural
def pre_processamento_texto(variavel):

    frases = list(variavel)
    
    textos_processados = []
    manter = {'não', 'nem', 'sem', 'mas', 'muito', 'mais', 'só', 'já', 'até', 'depois', 'também'}
    stop_words = set(nltk.corpus.stopwords.words('portuguese')) - manter
    
    for f in frases:
        palavras = re.sub(r'[^\w\s]', '', str(f).lower()) # limpa o texto, deixando somente letras, números e espaços, tudo em minúsculas.
        palavras = nltk.word_tokenize(palavras) # quebra o texto em tokens
        palavras_filtradas = [palavra for palavra in palavras if palavra not in stop_words]  
        palavras_filtradas = ' '.join(palavras_filtradas)
        textos_processados.append(palavras_filtradas)
        
    return pd.Series(textos_processados, index=variavel.index)
    
corpus = list(pre_processamento_texto(dados_limpos['review_comment_message'])) # Documento com os textos

# 2. Definindo rótulos
def rotular(var_score):
    if var_score >= 4:
        return 'Satisfeito'
    if var_score == 3:
        return 'Neutro'
    return 'Insatisfeito'

In [6]:
# 3. Definindo Processo dos dados
# x -> entrada -> matriz com os comentários vetorizados
# y -> saída -> rótulo de satisfação (satisfeito, neutro, insatisfeito)

x = corpus
y = dados_limpos['review_score'].apply(rotular)
print(len(corpus) == len(y))

# Divide os dados em treino e teste
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, shuffle=True, random_state=42)

# Etapas Pipeline
modelo = Pipeline([
    ('vetorizacao_1', CountVectorizer()), # Criação do modelo BoW (saco de palavras para vetorização)
    ('regressao_logistica', LogisticRegression(max_iter=1000, class_weight='balanced')) # Cria o modelo de regressão logística
])

modelo.fit(x_train,y_train) # Treinando
teste = modelo.predict(x_test) # Testando
teste

True


array(['Neutro', 'Neutro', 'Satisfeito', ..., 'Insatisfeito', 'Neutro',
       'Satisfeito'], dtype=object)

In [7]:
# 4. Validando o modelo -> holdout
print(classification_report(y_test, teste))
print(confusion_matrix(y_test, teste, labels=["Satisfeito", "Neutro", "Insatisfeito"]))

primeiro_resultado = pd.DataFrame({
    'comentario': x_test,
    'sentimento_real': y_test,
    'sentimento_previsto': teste
})
display(primeiro_resultado.head(30))
print((y_test == teste).value_counts())

              precision    recall  f1-score   support

Insatisfeito       0.78      0.78      0.78      2036
      Neutro       0.24      0.36      0.29       692
  Satisfeito       0.93      0.87      0.90      5200

    accuracy                           0.80      7928
   macro avg       0.65      0.67      0.66      7928
weighted avg       0.83      0.80      0.82      7928

[[4533  454  213]
 [ 208  249  235]
 [ 126  331 1579]]


,comentario,sentimento_real,sentimento_previsto
75882,ter mais cuiado hora embala transporte película chegou quebrada não gostei tudo bem comprar outra kit presente mim devolver não compensava,Satisfeito,Neutro
10425,nao recebi produto acredito problema correios prazo entrega expirou,Insatisfeito,Neutro
48346,ok recebido conforme informado,Satisfeito,Satisfeito
22064,ótimo produto linda chaleiracor muito linda,Satisfeito,Satisfeito
85902,recomendo,Satisfeito,Satisfeito
44844,comprei prancha profissional anti frizz 230 bivolte chegou cp9 profissional nao dizendo antifriz não gostei alisamento,Insatisfeito,Insatisfeito
93215,não recebi produto nem avisada falta item lojaquando entrei contatoquase 1 mês depoisé fiquei sabendo produto não entregue pois não havia mais estoque,Insatisfeito,Insatisfeito
33856,adorei,Satisfeito,Satisfeito
42507,tudo ok,Satisfeito,Satisfeito
12429,chegou prazo sem nenhuma discrepância,Satisfeito,Satisfeito


review_score
True     6361
False    1567
Name: count, dtype: int64


**Métricas de desempenho do modelo -> Validação holdout**

- Acurácia: o modelo acertou 80% de todas as previsões realizadas.
- Precisão: das previsões feitas para cada classe, a proporção que estava correta.
  - Satisfeito: 93%
  - Neutro: 24%
  - Insatisfeito: 78%
- Recall: dos casos que realmente pertencem a cada classe, a proporção que o modelo conseguiu identificar.
  - Satisfeito: 87%
  - Neutro: 36%
  - Insatisfeito: 78%

- Dos 5200 registros de teste que pertencem à classe 'Satisfeito': 4533 foram identificados pelo modelo como 'Satisfeito', 454 'Neutro' e 213 'Insatisfeito'.
- Dos 692 registros de teste que pertencem à classe 'Neutro': 249 foram identificados pelo modelo como 'Neutro', 208 'Satisfeito' e 235 'Insatisfeito'.
- Dos 2036 registros de teste que pertencem à classe 'Insatisfeito': 1579 foram identificados pelo modelo como 'Insatisfeito', 331 'Neutro' e 126 'Satisfeito'.

In [8]:
# 5. Validando o modelo -> validação cruzada

skfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42) # Objeto de divisão dos dados
y_previsto = cross_val_predict(modelo, x, y, cv=skfold)
print(classification_report(y, y_previsto))
print(confusion_matrix(y, y_previsto, labels=["Satisfeito", "Neutro", "Insatisfeito"]))

segundo_resultado = pd.DataFrame({
    'comentario': x,
    'sentimento_real': y,
    'sentimento_previsto': y_previsto
})
display(segundo_resultado.head(30))
print((y == y_previsto).value_counts())

              precision    recall  f1-score   support

Insatisfeito       0.77      0.77      0.77     10178
      Neutro       0.24      0.35      0.29      3463
  Satisfeito       0.93      0.87      0.90     25997

    accuracy                           0.80     39638
   macro avg       0.65      0.67      0.65     39638
weighted avg       0.83      0.80      0.82     39638

[[22725  2159  1113]
 [ 1026  1226  1211]
 [  603  1698  7877]]


,comentario,sentimento_real,sentimento_previsto
0,não testei produto ainda mas veio correto boas condições apenas caixa veio bem amassada danificada ficará chato pois trata presente,Satisfeito,Satisfeito
3,muito bom produto,Satisfeito,Satisfeito
5,produto exatamente esperava descrito site chegou bem antes data prevista,Satisfeito,Satisfeito
12,aguardando retorno loja,Insatisfeito,Insatisfeito
17,gostei produto,Satisfeito,Satisfeito
18,obrigado atenção lojas lannister perfeito tudo,Satisfeito,Satisfeito
24,correios estäo greve näo recebi nenhum aviso atraso mas qqer forma empresa responsável deveria dar alguma resposta sobre atraso,Insatisfeito,Insatisfeito
28,sempre vou comprar aqui pois melhor parabéns,Satisfeito,Satisfeito
29,demora muito entregar já passou prazo ainda não recebi encomenda,Insatisfeito,Insatisfeito
30,até momento não recebi produto protetor colchão berço algodão impermeável já fiz reclamação desde dia 1806 ainda não obtive retorno irei cancelar pedido estornar compra hj,Insatisfeito,Insatisfeito


review_score
True     31828
False     7810
Name: count, dtype: int64


**Métricas de desempenho do modelo -> Validação cruzada**

- Acurácia: o modelo acertou 80% de todas as previsões realizadas.
- Precisão: das previsões feitas para cada classe, a proporção que estava correta.
  - Satisfeito: 93%
  - Neutro: 24%
  - Insatisfeito: 77%
- Recall: dos casos que realmente pertencem a cada classe, a proporção que o modelo conseguiu identificar.
  - Satisfeito: 87%
  - Neutro: 35%
  - Insatisfeito: 77%


- Dos 25997 registros da base que pertencem à classe 'Satisfeito': 22725 foram identificados pelo modelo como 'Satisfeito', 2159 'Neutro' e 1113 'Insatisfeito'.
- Dos 3463 registros da base que pertencem à classe 'Neutro': 1226 foram identificados pelo modelo como 'Neutro', 1026 'Satisfeito' e 1211 'Insatisfeito'.
- Dos 10178 registros da base que pertencem à classe 'Insatisfeito': 7877 foram identificados pelo modelo como 'Insatisfeito', 1698 'Neutro' e 603 'Satisfeito'.

**Embora os resultados dos dois métodos de validação sejam bastante próximos, em ambos o modelo tem dificuldade em identificar clientes da classe Neutro (F1 de 0,29). Essa classe representa apenas cerca de 8,7% dos registros, o que dificulta o aprendizado do modelo, e além disso é naturalmente ambígua, pois os erros se distribuem entre Satisfeito e Insatisfeito.**

## Análises descritivas

In [9]:
# Comentários mais ditos -> Visão Macro
c = pd.Series(corpus)
frases_mais_ditas = c.value_counts()
frases_mais_ditas

muito bom                                                                                                                        595
bom                                                                                                                              361
recomendo                                                                                                                        325
otimo                                                                                                                            241
ótimo                                                                                                                            223
                                                                                                                                ... 
relógio recebido não condiz foto anúncio creio relógio falso entrei contato loja mas até agora não obtive retorno                  1
comprei relógio veio modelo diferente solicitei                      

In [10]:
# Palavras mais ditas -> Visão Macro
palavras = []
for texto in corpus:
    palavra = texto.split()
    for p in palavra:
        palavras.append(p)
palavras

p = pd.Series(palavras)
palavras_mais_ditas = p.value_counts()
palavras_mais_ditas

produto         17437
não             10022
prazo            8048
muito            7659
entrega          6220
                ...  
engual              1
jjjjjj              1
prazopoderia        1
defeitoem           1
enxofre             1
Name: count, Length: 18583, dtype: int64

In [11]:
# Maiores Scores -> Visão Macro
display(dados_limpos['review_score'].value_counts()) # Valor total
display(dados_limpos['review_score'].value_counts()/len(dados_limpos) * 100) # Proporção

review_score
5    20149
1     8118
4     5848
3     3463
2     2060
Name: count, dtype: int64

review_score
5    50.832534
1    20.480347
4    14.753519
3     8.736566
2     5.197033
Name: count, dtype: float64

In [12]:
# Categorias mais Avaliadas -> Visão Macro
display(dados_limpos['product_category_name'].value_counts()/len(dados_limpos)*100)

categoria_mesa_banho = dados_limpos[dados_limpos['product_category_name']== 'cama_mesa_banho']
display(categoria_mesa_banho['review_score'].value_counts()/len(categoria_mesa_banho) * 100) # Para cada nota de avaliação
display(categoria_mesa_banho)

# A categoria cama_mesa_banho é a que apresenta a maior quantidade de registros, 
# aproximadamente 10,6% do dataset. E dentro dessa categoria, há uma predominância de avaliações positivas (cerca de 62%), 
# mas há um valor consideravelmente alto para as notas baixas (cerca de 27%).

product_category_name
cama_mesa_banho                  10.626167
beleza_saude                      8.375801
esporte_lazer                     7.386851
informatica_acessorios            6.806600
moveis_decoracao                  6.352490
                                   ...    
fashion_esporte                   0.012614
cds_dvds_musicais                 0.012614
la_cuisine                        0.010091
seguros_e_servicos                0.005046
fashion_roupa_infanto_juvenil     0.002523
Name: count, Length: 71, dtype: float64

review_score
5    48.171890
1    20.892688
4    14.316239
3    10.375119
2     6.244065
Name: count, dtype: float64

,order_id,product_id,product_category_name,review_score,review_comment_message
135,8fc207e94fa91a7649c5a5dab690272a,f2e53dd1670f3c376518263b3f71424d,cama_mesa_banho,3,O prazo de entrega é muito longo
142,f2fa3795760c35b8c05fa8e2f09d5143,120fa011365fc39efe382cba4e50999e,cama_mesa_banho,3,Só foi entregue 1 dos 3 produtos q comprei
146,e37797aedc7cd4ef82278fbc169eecaf,013ee64977aaa6b2b25475095162e0e9,cama_mesa_banho,5,Muito bom entregou até antes do dia 😆😆😆😆😆😆😆😆
150,5164933efe0cd14f31424b9badf81f19,821a07e1e2a0382df1f3761ae3cd435c,cama_mesa_banho,1,Produto não corresponde às expectativas
187,1d067305b599c1e0dceb3864056ea527,2a2d22ae30e026f1893083c8405ca522,cama_mesa_banho,5,muito bom otimo
...,...,...,...,...,...
115532,2b9319f90cac6853394132c982ff27bf,2a2d22ae30e026f1893083c8405ca522,cama_mesa_banho,5,"Muito bom, muito confortável para dormir."
115538,a6bd1f93b7ff72cc348ca07f38ec4bee,c045b19fdb30a3035c56d04c6fe2e622,cama_mesa_banho,5,Foi tudo dentro do esperado!!!
115563,a3480694574e74545579bf7d974318d0,1a428b685ede76217c9efb550c4aaa59,cama_mesa_banho,5,"Gostei muito,super recomendo,recebi antes mesmo do prazo ."
115565,41d74f5b8f1fb05bdec674cbba8f720f,6fdbc2d597f906254d258d8d64d71925,cama_mesa_banho,1,Não gostei imaginei que era uma cortina de tecido bonita . E no entanto é nylon e muito mal acabada e frágil.


In [13]:
# O que fez 20,5% dos pedidos receberem nota 1(insatisfação)?
pior_avaliacao = dados_limpos[dados_limpos['review_score']==1]
pior_avaliacao

,order_id,product_id,product_category_name,review_score,review_comment_message
12,e6ce16cb79ec1d90b1da9085a6118aeb,08574b074924071f4e201e151b152b4e,ferramentas_jardim,1,Aguardando retorno da loja
30,acce194856392f074dbf9dada14d8d82,d70f38e7f79c630f8ea00c993897042c,bebes,1,"Até o momento não recebi o produto Protetor De Colchão Para Berço - Algodão Impermeável, já fiz a reclamação desde o dia 18/06 e ainda não obtive retorno. Irei cancelar o pedido e estornar a compra hj"
36,1790eea0b567cf50911c057cf20f90f9,2d8f2be4f08788ee3bf5356af2b2ee6c,climatizacao,1,"Foi entregue um produto divergente do que eu havia comprado, além de não vir com todos os acessórios constantes na descrição do anuncio. Solicito providências urgentes."
46,6ea2f835b4556291ffdc53fa0b3b95e8,be021417a6acb56b9b50d3fd2714baa8,utilidades_domesticas,1,"Inicialmente, na data da compra o produto era para ser entregue até 21/12. Hoje é dia 27/12 e não chegou ainda."
50,6ebaec694d7025e2ad4a05dba887c032,e251ebd2858be1aa7d9b2087a6992580,ferramentas_jardim,1,A cuba veio quebrada. Como efetuo barroca?
...,...,...,...,...,...
115560,0fa1fab1d7c1211c824596ed5e111e3c,149de4383ccb1829ea75380fc5f0b5fa,beleza_saude,1,"Nao volto a comprar com esta loja , demorando uma vida a enttega deste produto"
115565,41d74f5b8f1fb05bdec674cbba8f720f,6fdbc2d597f906254d258d8d64d71925,cama_mesa_banho,1,Não gostei imaginei que era uma cortina de tecido bonita . E no entanto é nylon e muito mal acabada e frágil.
115586,38e9133ce29f6bbe35aed9c3863dce01,014a8a503291921f7b004a5215bb3c36,bebes,1,nao recebi o produto que ja paguei ! nao recomendo a compra nesse site.
115588,d692ef54145c9cb3322ec2e5508aa3f4,58efb9b638561ce132216a9a612513e2,moveis_sala,1,"Prateleiras com cantos vivo, colunas nao encaixam, faltam arroelas para nao danificar a pintura, prateleiras chegaram todas amassadas."


In [14]:
display(pior_avaliacao['product_id'].value_counts()/len(pior_avaliacao) * 100) # Proporção produtos com nota 1 -> Visão micro
print(40*'-')
display(pior_avaliacao['product_category_name'].value_counts()/len(pior_avaliacao) * 100) # Proporção categorias com nota 1 -> Visão micro
print(40*'-')

# Percebe-se que qualidade do produto e prazo de entrega são temas recorrentes que podem justificar o recebimento da menor nota: 1 -> Insatisfeito -> visão micro
avaliacao = pior_avaliacao['review_comment_message']
avaliacao = pd.DataFrame(avaliacao)
avaliacao.head(30)


product_id
99a4788cb24856965c36a24e339b6058    0.468096
53759a2ecddad2bb87a079a1f1519f73    0.443459
89b121bee266dcd25688a1ba72eefb61    0.381867
7c1bd920dbdf22470b68bde975dd3ccf    0.344913
422879e10f46682990de24d770e7f83d    0.332594
                                      ...   
7e9b52ab8c9b4c0fa93e431e8c3975e4    0.012318
2610a3a6c0250a7f4411801a8cf376c7    0.012318
05f97ad4ce078ed1de8882e1824eb07f    0.012318
7e646f7e1e428a1ddde5a18dd9d42e95    0.012318
74f6a88474993377506706e383f98302    0.012318
Name: count, Length: 5472, dtype: float64

----------------------------------------


product_category_name
cama_mesa_banho           10.840108
informatica_acessorios     8.167036
beleza_saude               8.117763
esporte_lazer              7.538803
moveis_decoracao           7.156935
                            ...    
musica                     0.024637
casa_conforto_2            0.024637
la_cuisine                 0.012318
seguros_e_servicos         0.012318
fashion_esporte            0.012318
Name: count, Length: 69, dtype: float64

----------------------------------------


,review_comment_message
12,Aguardando retorno da loja
30,"Até o momento não recebi o produto Protetor De Colchão Para Berço - Algodão Impermeável, já fiz a reclamação desde o dia 18/06 e ainda não obtive retorno. Irei cancelar o pedido e estornar a compra hj"
36,"Foi entregue um produto divergente do que eu havia comprado, além de não vir com todos os acessórios constantes na descrição do anuncio. Solicito providências urgentes."
46,"Inicialmente, na data da compra o produto era para ser entregue até 21/12. Hoje é dia 27/12 e não chegou ainda."
50,A cuba veio quebrada. Como efetuo barroca?
57,"Recebi apenas 1 unidade solicitada, deveriam ser 2, inclusive foi cobrado e consta na nota fiscal. Estou tentando contato com a loja a 4 dias. Informam que vão me mandar por email o contato do fornece"
63,"Pedir 2 capinhas,estou pagando as duas e só veio uma.Entao não estou nada satisfeita e espero que seja reparado esse erro."
65,Não recebi
94,Muito bom o produto e preço cobrado!
111,"Os outros produtos, a loja ofereceu mas não cumpriu com a entrega."


**Conclusão:**

A análise descritiva mostra predominância de avaliações positivas: as notas 5 representam cerca de 50% dos 39638 registros. Somadas às notas 4, que correspondem ao sentimento Satisfeito, chegam a aproximadamente 65,6% do dataset. Por outro lado, as notas 1 representam cerca de 20,5% dos dados. Considerando também as notas 2, a classe Insatisfeito soma 25,7%.

Entre as avaliações nota 1, a categoria cama, mesa e banho concentra a maior parcela (cerca de 10,8%), proporção semelhante ao seu peso no dataset (cerca de 10,6%), o que indica que ela aparece mais por ser a categoria com mais avaliações. Os comentários abrangem desde problemas de qualidade do produto até relatos de falhas na entrega, como atrasos e pedidos que não chegaram. Esses resultados indicam a importância de avaliar a gestão de qualidade dos produtos e investigar as causas dos atrasos nas entregas, incluindo o desempenho das transportadoras. Também é importante ressaltar que dentro dessa categoria, no geral, há uma maior predominância de avaliações positivas, indicando que tais problemas são a minoria mas que requerem melhorias.

Modelo de classificação de sentimento:

Para o caso de comentários sem rótulo, treinei um modelo de regressão logística que prevê o sentimento a partir do texto, usando balanceamento de classes (class_weight='balanced'). O desempenho geral foi bom, com destaque para Satisfeito e Insatisfeito. A classe Neutro (nota 3) teve desempenho inferior, com queda na precisão e no recall, mesmo após o balanceamento.

## Perguntas à serem respondidas

1. Diante da base de dados utilizada (cite qual foi), qual seria um problema relevante e acionável a ser resolvido utilizando Ciência de Dados? Justifique sua escolha com base em possíveis impactos para os negócios, organizações e pessoas possivelmente interessadas e hipóteses levantadas a partir das variáveis disponíveis.

A base utilizada foi o Merged Olist Brazilian E-Commerce Dataset for EDA, disponível no Kaggle, que reúne dados de pedidos, produtos, entregas e avaliações de clientes de um e-commerce brasileiro.

Um problema relevante e acionável é: quando surgirem clientes cujas avaliações não vierem rotuladas (sem nota de satisfação), como identificar o nível de satisfação deles a partir do comentário que escreveram? Esse foi o problema que explorei no meu projeto.

Resolvê-lo traz impacto direto para o negócio. A empresa passa a monitorar a satisfação em larga escala e a entender como a gestão de recursos, os produtos e as transportadoras influenciam a percepção do cliente sobre o recebimento do pedido. Com as causas identificadas, ela pode agir para melhorar a experiência e reduzir a rotatividade de clientes. Vendedores, transportadoras e os próprios clientes também se beneficiam.

Para resolver o problema, iniciei meus estudos em aprendizado de máquina supervisionado e escolhi a regressão logística. Nela, uma variável de entrada (x), o texto do comentário, e rótulos de treinamento (y), a classificação de satisfação, permitem treinar o algoritmo para prever classes discretas para novos comentários

2. Quais ferramentas, bibliotecas e etapas você usou para explorar os dados, fazer tratamento, e criar visualizações úteis para investigar o problema formulado? Cite ao menos duas funções aprendidas nas últimas aulas (ex.: funções em Python, manipulação com Pandas, ou visualizações com matplotlib/seaborn) que seriam essenciais neste processo.

Utilizei Python em um notebook Jupyter, com as seguintes bibliotecas:

- pandas: tratamento dos dados e análise descritiva.
- re: expressões regulares no pré-processamento do texto.
- nltk: processamento de linguagem natural. Como o computador não interpreta palavras, precisei vetorizar o texto, transformando palavras em números.
- scikit-learn: treinamento e validação do modelo com os dados de teste.

As etapas foram: exploração e tratamento dos dados, pré-processamento do texto, vetorização, divisão em treino e teste, treinamento do modelo e avaliação dos resultados. As principais funções do scikit-learn foram:

- CountVectorizer(): cria o modelo Bag of Words para vetorizar o texto.
- LogisticRegression(max_iter=1000, class_weight='balanced'): cria o modelo de regressão logística, com pesos ajustados para compensar classes desbalanceadas.
- classification_report(): apresenta as métricas de desempenho do modelo.
- confusion_matrix(): mostra os acertos e erros de cada classe.

3. Suponha que, ao aplicar uma análise exploratória, você percebe que uma variável da sua base de dados apresenta assimetria à direita e muitos outliers. Como você interpretaria esse comportamento estatístico e como isso afetaria sua decisão ao propor soluções? Como comunicaria isso para uma equipe não técnica?

Uma assimetria à direita indica que a maior parte dos dados se concentra em valores baixos, enquanto os valores altos são mais raros e dispersos, formando uma cauda longa à direita. Com isso, a média é puxada para cima e fica acima da mediana, deixando de representar bem o valor típico.

Eu confirmaria os outliers por cálculo, usando o IQR e os limites inferior (Q1 − 1,5 × IQR) e superior (Q3 + 1,5 × IQR). Em seguida, seria meu papel como analista investigar a origem deles: podem ser situações raras, mas legítimas, medições incorretas, falhas de entrada ou anomalias. Ignorar essa limitação pode levar a análises imprecisas e a prejuízos financeiros e operacionais.

Isso também afetaria minhas decisões. Eu usaria a mediana em vez da média para descrever o valor típico, decidiria caso a caso entre manter, tratar ou remover os outliers.

Para uma equipe não técnica, eu diria: “Pessoal, a maior parte dos valores dessa variável é baixa, e a média acaba sendo puxada para cima por poucos casos muito altos. Por isso, o valor mais comum é menor do que a média sugere. Esses valores altos podem ser casos reais e raros ou erros de registro, e estou investigando para que as decisões de vocês sejam baseadas em informações mais precisas.”

4. Após finalizar o projeto, como você avalia a relevância dele para seu portfólio como Cientista de Dados? O que destacaria como diferencial técnico e como ponto de melhoria se tivesse que refazer o Projeto Prático III?

Considero este projeto fundamental para o início da minha trajetória em ciência de dados. Todos os projetos anteriores envolviam analisar dados do passado para compreender o presente e apoiar decisões. Neste, pela primeira vez, apliquei aprendizado de máquina para prever classificações em um contexto de e-commerce, usando a biblioteca scikit-learn. Ainda não domino todas as técnicas de machine learning nem todas as funções da biblioteca, mas estou satisfeita com esse começo em processamento de linguagem natural e regressão logística.

Como diferencial técnico, destaco o tratamento do desbalanceamento das classes com class_weight='balanced' e a avaliação do modelo por classification_report e matriz de confusão, e não apenas pela acurácia.

Se refizesse o projeto, aprimoraria o pré-processamento do texto, por exemplo removendo acentos, e testaria outras técnicas, como TF-IDF. Também incluiria visualizações, que não utilizei.
